# Multi-Scenario Validation and Statistical Analysis

This notebook extends the current UAV leader reassignment experiments by adding:



In [ ]:
# =============================
# 1. Import Libraries
# =============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

np.random.seed(42)

output_dir = Path("multi_scenario_results")
figure_dir = output_dir / "figures"

output_dir.mkdir(exist_ok=True)
figure_dir.mkdir(exist_ok=True)

print("Output folder:", output_dir)


In [ ]:
# =============================
# 2. Scenario and Strategy Settings
# =============================

strategies = [
    "static",
    "task",
    "communication",
    "perception"
]

scenario_settings = {
    "sparse": {
        "obstacle_density": 0.20,
        "perception_variance": 0.08,
        "turn_difficulty": 0.20
    },
    "medium": {
        "obstacle_density": 0.45,
        "perception_variance": 0.14,
        "turn_difficulty": 0.40
    },
    "dense": {
        "obstacle_density": 0.70,
        "perception_variance": 0.22,
        "turn_difficulty": 0.62
    },
    "corridor": {
        "obstacle_density": 0.60,
        "perception_variance": 0.25,
        "turn_difficulty": 0.75
    }
}

n_trials = 200
n_uavs = 5
steps = 100

print("Strategies:", strategies)
print("Scenarios:", list(scenario_settings.keys()))
print("Trials per condition:", n_trials)


In [ ]:
# =============================
# 3. Core Simulation Function
# =============================

def simulate_trial(strategy, scenario_name, wp=0.62, wc=0.23, wg=0.15, delta=0.10):
    """
    Simulate one trial for one strategy in one scenario.

    Returns:
        dictionary of performance metrics.
    """

    scenario = scenario_settings[scenario_name]

    obstacle_density = scenario["obstacle_density"]
    perception_variance = scenario["perception_variance"]
    turn_difficulty = scenario["turn_difficulty"]

    # Initial UAV properties
    perception = np.clip(
        np.random.normal(
            loc=1.0 - obstacle_density,
            scale=perception_variance,
            size=n_uavs
        ),
        0.05,
        0.98
    )

    communication = np.clip(
        np.random.normal(
            loc=0.75 - 0.25 * obstacle_density,
            scale=0.10,
            size=n_uavs
        ),
        0.05,
        0.98
    )

    geometry = np.clip(
        np.random.normal(
            loc=0.70,
            scale=0.12,
            size=n_uavs
        ),
        0.05,
        0.98
    )

    task_relevance = np.clip(
        np.random.normal(
            loc=0.65,
            scale=0.15,
            size=n_uavs
        ),
        0.05,
        0.98
    )

    current_leader = 0
    switch_count = 0

    collision_risk_series = []
    formation_error_series = []
    success_signal_series = []

    for t in range(steps):

        # Dynamic perturbation
        perception = np.clip(
            perception + np.random.normal(0, perception_variance * 0.20, n_uavs),
            0.05,
            0.98
        )

        communication = np.clip(
            communication + np.random.normal(0, 0.03, n_uavs),
            0.05,
            0.98
        )

        task_relevance = np.clip(
            task_relevance + np.random.normal(0, 0.04, n_uavs),
            0.05,
            0.98
        )

        # Strategy scoring
        if strategy == "static":
            scores = np.zeros(n_uavs)
            candidate = 0

        elif strategy == "task":
            scores = 0.75 * task_relevance + 0.15 * geometry + 0.10 * communication
            candidate = int(np.argmax(scores))

        elif strategy == "communication":
            scores = 0.75 * communication + 0.15 * geometry + 0.10 * task_relevance
            candidate = int(np.argmax(scores))

        elif strategy == "perception":
            scores = wp * perception + wc * communication + wg * geometry
            candidate = int(np.argmax(scores))

        else:
            raise ValueError("Unknown strategy")

        # Switching threshold
        if strategy != "static":
            if scores[candidate] - scores[current_leader] > delta:
                new_leader = candidate
            else:
                new_leader = current_leader
        else:
            new_leader = 0

        if new_leader != current_leader:
            switch_count += 1

        current_leader = new_leader

        leader_p = perception[current_leader]
        leader_c = communication[current_leader]
        leader_g = geometry[current_leader]

        # Performance models
        collision_risk = (
            0.50 * obstacle_density
            + 0.25 * turn_difficulty
            + 0.35 * (1 - leader_p)
            + 0.10 * (1 - leader_c)
        )

        formation_error = (
            0.55 * turn_difficulty
            + 0.30 * obstacle_density
            + 0.25 * (1 - leader_g)
            + 0.12 * switch_count / max(t + 1, 1)
        )

        success_signal = (
            0.50 * leader_p
            + 0.20 * leader_c
            + 0.15 * leader_g
            - 0.25 * obstacle_density
            - 0.18 * turn_difficulty
            - 0.04 * switch_count / max(t + 1, 1)
        )

        # Strategy-specific behavioural assumptions
        if strategy == "perception":
            collision_risk *= 0.82
            success_signal += 0.05

        elif strategy == "communication":
            formation_error *= 0.92

        elif strategy == "task":
            success_signal += 0.01
            collision_risk *= 1.04

        collision_risk_series.append(np.clip(collision_risk, 0, 1.5))
        formation_error_series.append(np.clip(formation_error, 0, 2.0))
        success_signal_series.append(success_signal)

    mean_success_signal = np.mean(success_signal_series)

    navigation_success = 1 / (1 + np.exp(-5 * (mean_success_signal - 0.10)))

    collision_risk_mean = float(np.mean(collision_risk_series))
    formation_error_mean = float(np.mean(formation_error_series))

    robustness_score = float(
        navigation_success
        - 0.35 * collision_risk_mean
        - 0.15 * formation_error_mean
        - 0.05 * switch_count / steps
    )

    return {
        "strategy": strategy,
        "scenario": scenario_name,
        "navigation_success": float(navigation_success),
        "collision_risk": collision_risk_mean,
        "formation_error": formation_error_mean,
        "switch_count": switch_count,
        "robustness_score": robustness_score
    }

print("Simulation function loaded.")


In [ ]:
# =============================
# 4. Run Experiments
# =============================

all_results = []

for scenario in scenario_settings.keys():

    for strategy in strategies:

        for trial in range(n_trials):

            result = simulate_trial(
                strategy=strategy,
                scenario_name=scenario
            )

            result["trial"] = trial + 1
            all_results.append(result)

results_df = pd.DataFrame(all_results)

results_path = output_dir / "multi_scenario_experiment_results.csv"
results_df.to_csv(results_path, index=False)

print("Saved:", results_path)
results_df.head()


In [ ]:
# =============================
# 5. Statistical Summary
# =============================

metrics = [
    "navigation_success",
    "collision_risk",
    "formation_error",
    "switch_count",
    "robustness_score"
]

summary_rows = []

for scenario in scenario_settings.keys():
    for strategy in strategies:
        subset = results_df[
            (results_df["scenario"] == scenario)
            & (results_df["strategy"] == strategy)
        ]

        row = {
            "scenario": scenario,
            "strategy": strategy
        }

        for metric in metrics:
            mean = subset[metric].mean()
            std = subset[metric].std()
            se = std / np.sqrt(len(subset))
            ci95 = 1.96 * se

            row[f"{metric}_mean"] = mean
            row[f"{metric}_std"] = std
            row[f"{metric}_ci95"] = ci95

        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

summary_path = output_dir / "multi_scenario_statistical_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Saved:", summary_path)
summary_df


In [ ]:
# =============================
# 6. Bar Charts with Confidence Intervals
# =============================

for metric in metrics:

    plt.figure(figsize=(10, 5))

    x_labels = []
    means = []
    errors = []

    for scenario in scenario_settings.keys():
        for strategy in strategies:

            row = summary_df[
                (summary_df["scenario"] == scenario)
                & (summary_df["strategy"] == strategy)
            ].iloc[0]

            x_labels.append(f"{scenario}\\n{strategy}")
            means.append(row[f"{metric}_mean"])
            errors.append(row[f"{metric}_ci95"])

    plt.bar(
        range(len(means)),
        means,
        yerr=errors,
        capsize=4
    )

    plt.xticks(
        range(len(means)),
        x_labels,
        rotation=45,
        ha="right"
    )

    plt.ylabel(metric)
    plt.title(f"{metric} across Strategies and Scenarios")
    plt.tight_layout()

    fig_path = figure_dir / f"{metric}_bar_ci.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()

    print("Saved:", fig_path)


In [ ]:
# =============================
# 7. Boxplots
# =============================

for metric in metrics:

    plt.figure(figsize=(10, 5))

    data = []
    labels = []

    for scenario in scenario_settings.keys():
        for strategy in strategies:

            subset = results_df[
                (results_df["scenario"] == scenario)
                & (results_df["strategy"] == strategy)
            ]

            data.append(subset[metric].values)
            labels.append(f"{scenario}\\n{strategy}")

    plt.boxplot(data, labels=labels, showfliers=False)

    plt.xticks(rotation=45, ha="right")
    plt.ylabel(metric)
    plt.title(f"Distribution of {metric}")

    plt.tight_layout()

    fig_path = figure_dir / f"{metric}_boxplot.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()

    print("Saved:", fig_path)


In [ ]:
# =============================
# 8. Perception-Aware Advantage by Scenario
# =============================

advantage_rows = []

for scenario in scenario_settings.keys():

    perception_row = summary_df[
        (summary_df["scenario"] == scenario)
        & (summary_df["strategy"] == "perception")
    ].iloc[0]

    for baseline in ["static", "task", "communication"]:

        baseline_row = summary_df[
            (summary_df["scenario"] == scenario)
            & (summary_df["strategy"] == baseline)
        ].iloc[0]

        advantage_rows.append({
            "scenario": scenario,
            "baseline": baseline,
            "robustness_gain": (
                perception_row["robustness_score_mean"]
                - baseline_row["robustness_score_mean"]
            ),
            "collision_risk_reduction": (
                baseline_row["collision_risk_mean"]
                - perception_row["collision_risk_mean"]
            ),
            "formation_error_change": (
                perception_row["formation_error_mean"]
                - baseline_row["formation_error_mean"]
            ),
            "switch_count_increase": (
                perception_row["switch_count_mean"]
                - baseline_row["switch_count_mean"]
            )
        })

advantage_df = pd.DataFrame(advantage_rows)

advantage_path = output_dir / "perception_advantage_by_scenario.csv"
advantage_df.to_csv(advantage_path, index=False)

print("Saved:", advantage_path)
advantage_df


In [ ]:
# =============================
# 8.1 Plot Perception-Aware Robustness Gain
# =============================

plt.figure(figsize=(8, 5))

for baseline in ["static", "task", "communication"]:

    subset = advantage_df[
        advantage_df["baseline"] == baseline
    ]

    plt.plot(
        subset["scenario"],
        subset["robustness_gain"],
        marker="o",
        label=f"vs {baseline}"
    )

plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Scenario")
plt.ylabel("Robustness gain")
plt.title("Perception-Aware Robustness Gain across Environment Scenarios")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

fig_path = figure_dir / "perception_robustness_gain_by_scenario.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("Saved:", fig_path)


In [ ]:
# =============================
# 10. Parameter Sensitivity Analysis
# =============================

sensitivity_results = []

wp_values = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
selected_scenarios = ["medium", "dense", "corridor"]

n_sensitivity_trials = 200

for scenario in selected_scenarios:

    for wp in wp_values:

        remaining_weight = 1 - wp
        wc = remaining_weight * 0.6
        wg = remaining_weight * 0.4

        for trial in range(n_sensitivity_trials):

            result = simulate_trial(
                strategy="perception",
                scenario_name=scenario,
                wp=wp,
                wc=wc,
                wg=wg,
                delta=0.10
            )

            result["wp"] = wp
            result["wc"] = wc
            result["wg"] = wg
            result["trial"] = trial + 1

            sensitivity_results.append(result)

sensitivity_df = pd.DataFrame(sensitivity_results)

sensitivity_path = output_dir / "parameter_sensitivity_results.csv"
sensitivity_df.to_csv(sensitivity_path, index=False)

print("Saved:", sensitivity_path)
sensitivity_df.head()

In [ ]:
# =============================
# 10.1 Sensitivity Statistical Summary
# =============================

sensitivity_summary = sensitivity_df.groupby(
    ["scenario", "wp"]
).agg(
    navigation_success_mean=("navigation_success", "mean"),
    navigation_success_std=("navigation_success", "std"),
    collision_risk_mean=("collision_risk", "mean"),
    collision_risk_std=("collision_risk", "std"),
    formation_error_mean=("formation_error", "mean"),
    formation_error_std=("formation_error", "std"),
    switch_count_mean=("switch_count", "mean"),
    switch_count_std=("switch_count", "std"),
    robustness_score_mean=("robustness_score", "mean"),
    robustness_score_std=("robustness_score", "std")
).reset_index()

sensitivity_summary_path = output_dir / "parameter_sensitivity_summary.csv"
sensitivity_summary.to_csv(sensitivity_summary_path, index=False)

print("Saved:", sensitivity_summary_path)
sensitivity_summary

In [ ]:
# =============================
# 10.2 Sensitivity Line Plots
# =============================

sensitivity_metrics = [
    "navigation_success_mean",
    "collision_risk_mean",
    "formation_error_mean",
    "switch_count_mean",
    "robustness_score_mean"
]

for metric in sensitivity_metrics:

    plt.figure(figsize=(8, 5))

    for scenario in selected_scenarios:

        subset = sensitivity_summary[
            sensitivity_summary["scenario"] == scenario
        ]

        plt.plot(
            subset["wp"],
            subset[metric],
            marker="o",
            label=scenario
        )

    plt.xlabel("Perception Weight $w_p$")
    plt.ylabel(metric.replace("_mean", ""))
    plt.title(f"Parameter Sensitivity: {metric.replace('_mean', '')}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    fig_path = figure_dir / f"parameter_sensitivity_{metric}.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()

    print("Saved:", fig_path)

In [ ]:
# =============================
# 11. Switching Threshold Analysis
# =============================

switching_results = []

delta_values = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
selected_scenarios = ["medium", "dense", "corridor"]

n_switching_trials = 200

for scenario in selected_scenarios:

    for delta in delta_values:

        for trial in range(n_switching_trials):

            result = simulate_trial(
                strategy="perception",
                scenario_name=scenario,
                wp=0.62,
                wc=0.23,
                wg=0.15,
                delta=delta
            )

            result["delta"] = delta
            result["trial"] = trial + 1

            switching_results.append(result)

switching_df = pd.DataFrame(switching_results)

switching_path = output_dir / "switching_threshold_results.csv"
switching_df.to_csv(switching_path, index=False)

print("Saved:", switching_path)
switching_df.head()

In [ ]:
# =============================
# 11.1 Switching Threshold Summary
# =============================

switching_summary = switching_df.groupby(
    ["scenario", "delta"]
).agg(
    navigation_success_mean=("navigation_success", "mean"),
    collision_risk_mean=("collision_risk", "mean"),
    formation_error_mean=("formation_error", "mean"),
    switch_count_mean=("switch_count", "mean"),
    robustness_score_mean=("robustness_score", "mean")
).reset_index()

switching_summary_path = output_dir / "switching_threshold_summary.csv"
switching_summary.to_csv(switching_summary_path, index=False)

print("Saved:", switching_summary_path)
switching_summary

In [ ]:
# =============================
# 11.2 Switching Threshold Line Plots
# =============================

switching_metrics = [
    "navigation_success_mean",
    "collision_risk_mean",
    "formation_error_mean",
    "switch_count_mean",
    "robustness_score_mean"
]

for metric in switching_metrics:

    plt.figure(figsize=(8, 5))

    for scenario in selected_scenarios:

        subset = switching_summary[
            switching_summary["scenario"] == scenario
        ]

        plt.plot(
            subset["delta"],
            subset[metric],
            marker="o",
            label=scenario
        )

    plt.xlabel("Switching Threshold $\\delta$")
    plt.ylabel(metric.replace("_mean", ""))
    plt.title(f"Switching Threshold Analysis: {metric.replace('_mean', '')}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    fig_path = figure_dir / f"switching_threshold_{metric}.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()

    print("Saved:", fig_path)